In [ ]:
from adaptive_latents.stim_designer import StimDesigner, OptimizationMethod, time, numpy, jnp, ScipyBoundedMinimize
from adaptive_latents.regressions import BaseMultiKernelRegressor
from tqdm.autonotebook import tqdm
import matplotlib.pyplot as plt
from copy import deepcopy

rng = numpy.random.default_rng()

In [ ]:
class SD(StimDesigner):
    def design_stim_prev_seen(self, v, previous_us, u_to_s_function=None):
        if u_to_s_function is None:
            u_to_s_function = lambda u: u

        def objective(u):
            s = u_to_s_function(u)
            s_norm = jnp.linalg.norm(s)
            loss = 0
            # loss += self.lam_1 * (self.max_l0_norm - jnp.sum(jnp.abs(u)))
            loss += jnp.dot(s, v) / (s_norm + 1e-10)
            return -loss.reshape()

        best_u = None
        best_loss = float('inf')
        for u in previous_us:
            loss = objective(u)
            if loss < best_loss:
                best_loss = loss
                best_u = u

        best_u = best_u / best_u.max()
        return best_u

    def design_stim_jaxopt(self, v, u_dimension, u_to_s_function=None):
        if u_to_s_function is None:
            u_to_s_function = lambda x: x

        u = self.rng.uniform(size=(u_dimension,)) * .1

        def objective(u):
            s = u_to_s_function(u)
            s_norm = jnp.linalg.norm(s)
            loss = self.lam_1 * (self.max_l0_norm - jnp.sum(jnp.abs(u)))
            loss += jnp.dot(s, v) / (s_norm + 1e-10)
            return -loss.reshape()

        lb = jnp.zeros_like(u)
        ub = jnp.ones_like(u)

        bounds = (lb, ub)
        intermediate_xs = []
        runner = ScipyBoundedMinimize(fun=objective, method='l-bfgs-b', callback=lambda xk: intermediate_xs.append(xk) if self.should_log else None)
        result = runner.run(u, bounds=bounds)
        u = numpy.array(result.params)

        if u.max() > 0:
            u = numpy.array(u / u.max())


        idx = numpy.argsort(u)
        u[idx[:-self.max_l0_norm]] = 0

        return u, {'s': u_to_s_function(u), 'intermediate_xs': numpy.array(intermediate_xs)}


    def design_stim(self, v, **kwargs):
        start_time = time.time()
        assert len(v.shape) == 2

        l = {}
        match self.optimization_method:
            case OptimizationMethod.JAXOPT:
                u, l = self.design_stim_jaxopt(v, kwargs['u_dimension'], kwargs['u_to_s_function'])
            case OptimizationMethod.CHEAT_LOWD_VEC:
                u = (kwargs['equivalent_projection_matrix'] @ v).flatten()
            case OptimizationMethod.CHEAT_HIGHD_VEC_SINGLE_NEURONS:
                u = numpy.zeros(kwargs['equivalent_projection_matrix'].shape[0])
                u[self.rng.choice(kwargs['equivalent_projection_matrix'].shape[0])] = 1
            case OptimizationMethod.CHEAT_HIGHD_VEC_MANY_NEURONS:
                u = numpy.zeros(kwargs['equivalent_projection_matrix'].shape[0])
                u[self.rng.choice(kwargs['equivalent_projection_matrix'].shape[0], size=self.max_l0_norm, replace=False)] = 1
            case "prev_seen":
                u = self.design_stim_prev_seen(v, previous_us=kwargs['previous_us'], u_to_s_function=kwargs['u_to_s_function'])
            case _:
                raise ValueError()


        if self.should_log:
            self.log.append({
                                'optimization_time': time.time() - start_time,
                                'v':v,
                                'u':u,
                                's': numpy.nan * v
                            } | l)

        return u

sd1 = SD(optimization_method="prev_seen")
sd2 = SD(optimization_method=OptimizationMethod.JAXOPT)



In [ ]:
stim_reg = BaseMultiKernelRegressor(maxlen=500)

def generate_observation(rng):
    u = rng.normal(size=3)
    u = numpy.abs(u)
    # u = u / u.max()
    state = rng.normal(size=3)
    t = numpy.array(rng.uniform(low=0, high=1))
    s = u
    return state, u, t, s

test_points = [generate_observation(rng) for _ in range(100)]
def assess_fit(stim_reg):
    errors = []
    for state, u, t, s in test_points:
        pred = stim_reg.predict([state, u, t])
        errors.append(pred - s)
    return errors

errors = []
for t in tqdm(numpy.arange(500)):
    state, u, t, s = generate_observation(rng)
    stim_reg.observe([state, u, t], s)
    errors.append(assess_fit(stim_reg))


In [ ]:
%matplotlib inline
errors = numpy.array(errors)

plt.plot(numpy.linalg.norm(errors, axis=-1));

In [ ]:
fig, ax = plt.subplots()
stim_reg.plot_length_scales(ax)

In [ ]:
l1, l2 = [], []

sd2.lam_1 = 1e10
for _ in tqdm(range(100)):
    state, _, t, _ = generate_observation(rng)
    u_to_s_function = lambda u: stim_reg.make_jax_pred_f()([state, u, t])
    v = numpy.abs(rng.normal(size=(3,1)))
    v = v / numpy.linalg.norm(v)


    u = sd1.design_stim(v, previous_us=stim_reg.input_histories[1][:stim_reg.n_observed], u_to_s_function=u_to_s_function)
    # u = sd2.design_stim(v, u_dimension=3,  u_to_s_function=lambda u: stim_reg.make_jax_pred_f()([state, u, t], length_scales=[1e-10, stim_reg.length_scales[1]/10, 1e-10]))
    # u = sd2.design_stim(v, u_dimension=3,  u_to_s_function=u_to_s_function)
    s = u_to_s_function(u)
    l1.append(s / numpy.linalg.norm(s) @ v)


    u = sd2.design_stim(v, u_dimension=3,  u_to_s_function=u_to_s_function)
    s = u_to_s_function(u)
    l2.append(s / numpy.linalg.norm(s) @ v)



In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))

ax.plot(l1,l2, '.')
ax.set_xlim(0,1)
ax.set_ylim(0,1);
